In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import os

# TASK 1. DATA DISCOVERY

In [ ]:
# Construct the raw GitHub URL for the .csv.gz file (this is currently from my drive right now since the dataset file is broken in github)
file_id = '1YMWXz-hnnJVGFaP5ZFNBSk5Q-YiGF00J'
github_url = f'https://drive.google.com/uc?id={file_id}&export=download'

# Load the gzipped CSV file directly into a pandas DataFrame
kidney_waitlist_analytics = pd.read_csv(github_url)

# Display the first 5 rows of the DataFrame
display(kidney_waitlist_analytics.head())

/tmp/ipykernel_2304/1387566348.py:6: DtypeWarning: Columns (24,25,26,28,29,32) have mixed types. Specify dtype option on import or set low_memory=False.
  kidney_waitlist_analytics = pd.read_csv(github_url)


,WL_ORG,ON_DIALYSIS,A2A2B_ELIGIBILITY,GENDER,ABO,BMI_TCR,FUNC_STAT_TCR,INIT_STAT,INIT_CPRA,END_CPRA,...,PSTATUS,PTIME,TRR_ID_CODE,DONOR_ID,LISTING_CTR_CODE,outcome,event_adverse,event_transplant,censored,days_to_event
0,KI,Y,NaN,F,B,31.6349,2080.0,4099.0,0.0,0.0,...,NaN,NaN,NaN,NaN,13609,died,1,0,0,287.0
1,KI,Y,NaN,M,A,30.0427,2070.0,4099.0,0.0,NaN,...,NaN,NaN,NaN,NaN,6975,removed_administrative,0,0,0,2322.0
2,KI,N,NaN,F,O,32.8515,2070.0,4099.0,0.0,0.0,...,NaN,NaN,NaN,NaN,19716,died,1,0,0,604.0
3,KI,Y,NaN,M,A,19.9982,2090.0,4099.0,0.0,0.0,...,NaN,NaN,NaN,NaN,8587,removed_too_sick,1,0,0,909.0
4,KI,Y,NaN,M,AB,23.2970,2070.0,4010.0,0.0,0.0,...,NaN,NaN,NaN,NaN,18352,removed_too_sick,1,0,0,231.0


## **A. Dataset Shape**

Expected: 494,862 rows × 40 columns

Cohort: WL_ORG == KI: waiting organ = kidney

INIT_DATE >= 2015-01-01: Earliest recording date = 2015-01-01

In [ ]:
# ============================================================
# A. Dataset Shape
# ============================================================

print("=== A. DATASET SHAPE ===")
print(f"Rows:    {kidney_waitlist_analytics.shape[0]:,}  (expected: 494,862)")
print(f"Columns: {kidney_waitlist_analytics.shape[1]}       (expected: 40)")
print(f"Row match:    {'✅' if kidney_waitlist_analytics.shape[0] == 494862 else '❌'}")
print(f"Column match: {'✅' if kidney_waitlist_analytics.shape[1] == 40 else '❌'}")

# Confirm cohort filters
print(f"\nWL_ORG unique values: {kidney_waitlist_analytics['WL_ORG'].unique()}")
print(f"INIT_DATE min: {kidney_waitlist_analytics['INIT_DATE'].min()}")

=== A. DATASET SHAPE ===
Rows:    494,862  (expected: 494,862)
Columns: 40       (expected: 40)
Row match:    ✅
Column match: ✅

WL_ORG unique values: ['KI']
INIT_DATE min: 2015-01-01


## **B. Outcome Distribution Checks**
Based on the data dictionary, the outcomes (or labels) we want to predict include:

1. 'transplanted'   
2. 'still_waiting'
3. 'removed_administrative'
4. 'transplanted_elsewhere'
5. 'removed_too_sick'
6. 'died'
7. 'unknown'

Note that when predicting, these outcomes will not be filled my the LLM, but rather, it would be predicted in certain codes that we would map to specific outcomes (see the data_dictionary for more details)

In [ ]:
# ============================================================
# B. Outcome Distribution
# Compare against extract_summary.txt counts exactly
# ============================================================

print("=== B. OUTCOME DISTRIBUTION ===\n")

expected_outcomes = {
    'transplanted':            234_429,
    'still_waiting':           103_300,
    'removed_administrative':   45_332,
    'transplanted_elsewhere':   37_243,
    'removed_too_sick':         35_219,
    'died':                     34_524,
    'unknown':                   4_815,
}

actual = kidney_waitlist_analytics['outcome'].value_counts()

for outcome, expected_count in expected_outcomes.items():
    actual_count = actual.get(outcome, 0)
    match = '✅' if actual_count == expected_count else '❌'
    pct = actual_count / len(kidney_waitlist_analytics) * 100
    print(f"{match} {outcome:<25} actual={actual_count:>8,}  expected={expected_count:>8,}  ({pct:.1f}%)")

# event_adverse check
adverse_sum = kidney_waitlist_analytics['event_adverse'].sum()
print(f"\n{'✅' if adverse_sum == 69743 else '❌'} event_adverse sum: {adverse_sum:,}  (expected: 69,743 = died + removed_too_sick)")
print(f"   event_adverse rate: {kidney_waitlist_analytics['event_adverse'].mean() * 100:.1f}%  (expected: 14.1%)")

=== B. OUTCOME DISTRIBUTION ===

✅ transplanted              actual= 234,429  expected= 234,429  (47.4%)
✅ still_waiting             actual= 103,300  expected= 103,300  (20.9%)
✅ removed_administrative    actual=  45,332  expected=  45,332  (9.2%)
✅ transplanted_elsewhere    actual=  37,243  expected=  37,243  (7.5%)
✅ removed_too_sick          actual=  35,219  expected=  35,219  (7.1%)
✅ died                      actual=  34,524  expected=  34,524  (7.0%)
✅ unknown                   actual=   4,815  expected=   4,815  (1.0%)

✅ event_adverse sum: 69,743  (expected: 69,743 = died + removed_too_sick)
   event_adverse rate: 14.1%  (expected: 14.1%)


## **C. Data Missingness Checks**

In [ ]:
# ============================================================
# C. Missingness
# Compare against extract_summary.txt percentages
# ============================================================

print("=== C. MISSINGNESS ===\n")

expected_missing = {
    'WLKI':                 100.00,
    'MULTIORG':              97.60,
    'A2A2B_ELIGIBILITY':     92.27,
    'COMPOSITE_DEATH_DATE':  80.53,
    'PTIME':                 54.03,
    'DIAG_KI':               53.66,
    'DON_TY':                53.37,
    'DONOR_ID':              53.37,
    'TRR_ID_CODE':           53.37,
    'PREV_TX':               53.37,
    'TX_DATE':               53.37,
    'PSTATUS':               53.37,
    'ORGAN':                 53.37,
    'INIT_CPRA':             28.84,
    'END_CPRA':              28.79,
    'DIALYSIS_DATE':         27.13,
    'REM_CD':                20.87,
    'DAYSWAIT_ALLOC':         1.36,
    'FUNC_STAT_TCR':          0.80,
    'BMI_TCR':                0.31,
    'days_to_event':          0.01,
    'DAYSWAIT_CHRON':         0.01,
}

zero_missing = [
    'INIT_STAT', 'censored', 'event_transplant', 'event_adverse',
    'outcome', 'LISTING_CTR_CODE', 'GENDER', 'ABO', 'ON_DIALYSIS',
    'REGION', 'END_STAT', 'INIT_AGE', 'END_DATE', 'INIT_DATE',
    'ETHCAT', 'PT_CODE', 'WL_ID_CODE', 'WL_ORG',
]

actual_missing = (kidney_waitlist_analytics.isnull().sum() / len(kidney_waitlist_analytics) * 100).round(2)

for col, expected_pct in expected_missing.items():
    if col not in kidney_waitlist_analytics.columns:
        print(f"⚠️  {col:<25} NOT IN DATAFRAME")
        continue
    actual_pct = actual_missing[col]
    match = '✅' if abs(actual_pct - expected_pct) < 0.05 else '❌'
    print(f"{match} {col:<25} actual={actual_pct:>6.2f}%  expected={expected_pct:>6.2f}%")

print()

zero_fails = []
for col in zero_missing:
    if col not in kidney_waitlist_analytics.columns:
        zero_fails.append(f"{col} (NOT IN DATAFRAME)")
        continue
    if actual_missing[col] > 0.00:
        zero_fails.append(f"{col} ({actual_missing[col]:.2f}%)")

if zero_fails:
    print(f"❌ Zero-missingness failures: {zero_fails}")
else:
    print(f"✅ All {len(zero_missing)} zero-missingness columns confirmed at 0.00%")

=== C. MISSINGNESS ===

✅ WLKI                      actual=100.00%  expected=100.00%
✅ MULTIORG                  actual= 97.60%  expected= 97.60%
✅ A2A2B_ELIGIBILITY         actual= 92.27%  expected= 92.27%
✅ COMPOSITE_DEATH_DATE      actual= 80.53%  expected= 80.53%
✅ PTIME                     actual= 54.03%  expected= 54.03%
✅ DIAG_KI                   actual= 53.66%  expected= 53.66%
✅ DON_TY                    actual= 53.37%  expected= 53.37%
✅ DONOR_ID                  actual= 53.37%  expected= 53.37%
✅ TRR_ID_CODE               actual= 53.37%  expected= 53.37%
✅ PREV_TX                   actual= 53.37%  expected= 53.37%
✅ TX_DATE                   actual= 53.37%  expected= 53.37%
✅ PSTATUS                   actual= 53.37%  expected= 53.37%
✅ ORGAN                     actual= 53.37%  expected= 53.37%
✅ INIT_CPRA                 actual= 28.84%  expected= 28.84%
✅ END_CPRA                  actual= 28.79%  expected= 28.79%
✅ DIALYSIS_DATE             actual= 27.13%  expected= 27.13%


## **D. Features Validity Checks**

Based on the finding above, we confirm that the input data from the dataset actually match with the column manifest from our project.

Next, I will then attempt to check all the features data entry to see if they are actually usable for further modeling.

This process is done mainly by finding imposible values and distributions shapes in each features.


In [ ]:
# ============================================================
# D1. Numerical features — INIT_AGE, INIT_CPRA, BMI_TCR
# Check for impossible values and distribution shape
# ============================================================

print("=== D1. NUMERICAL FEATURES ===\n")

# INIT_AGE
print("--- INIT_AGE ---")
print(kidney_waitlist_analytics['INIT_AGE'].describe())
age = kidney_waitlist_analytics['INIT_AGE']
print(f"\nNegative ages:  {(age < 0).sum()}")
print(f"Ages > 100:    {(age > 100).sum()}")
print(f"Ages == 0:     {(age == 0).sum()}")

# INIT_CPRA
print("\n--- INIT_CPRA ---")
print(kidney_waitlist_analytics['INIT_CPRA'].describe())
cpra = kidney_waitlist_analytics['INIT_CPRA'].dropna()
print(f"\nValues < 0:    {(cpra < 0).sum()}")
print(f"Values > 100:  {(cpra > 100).sum()}")

# BMI_TCR
print("\n--- BMI_TCR ---")
print(kidney_waitlist_analytics['BMI_TCR'].describe())
bmi = kidney_waitlist_analytics['BMI_TCR'].dropna()
print(f"\nBMI < 10 (implausible):    {(bmi < 10).sum()}")
print(f"BMI > 70 (implausible):    {(bmi > 70).sum()}")
print(f"BMI > 100 (extreme):       {(bmi > 100).sum()}")

=== D1. NUMERICAL FEATURES ===

--- INIT_AGE ---
count    494862.000000
mean         51.874533
std          14.601018
min           0.000000
25%          43.000000
50%          54.000000
75%          63.000000
max          91.000000
Name: INIT_AGE, dtype: float64

Negative ages:  0
Ages > 100:    0
Ages == 0:     123

--- INIT_CPRA ---
count    352121.000000
mean          9.786141
std          24.799960
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max         100.000000
Name: INIT_CPRA, dtype: float64

Values < 0:    0
Values > 100:  0

--- BMI_TCR ---
count    493304.000000
mean         64.700064
std        3141.118208
min           0.177300
25%          24.649900
50%          28.530350
75%          32.850400
max      430226.666700
Name: BMI_TCR, dtype: float64

BMI < 10 (implausible):    240
BMI > 70 (implausible):    368
BMI > 100 (extreme):       263


In [ ]:
# ============================================================
# D2. Categorical features — value counts for each
# Looking for unexpected codes and class imbalance
# ============================================================

print("=== D2. CATEGORICAL FEATURES ===\n")

categoricals = ['GENDER', 'ETHCAT', 'ABO', 'ON_DIALYSIS', 'INIT_STAT', 'REGION']

for col in categoricals:
    print(f"--- {col} ---")
    vc = kidney_waitlist_analytics[col].value_counts(dropna=False)
    pct = kidney_waitlist_analytics[col].value_counts(normalize=True, dropna=False) * 100
    summary = pd.DataFrame({'count': vc, 'pct': pct.round(2)})
    print(summary)
    print()

=== D2. CATEGORICAL FEATURES ===

--- GENDER ---
         count    pct
GENDER               
M       306408  61.92
F       188454  38.08

--- ETHCAT ---
         count    pct
ETHCAT               
1       202625  40.95
2       144926  29.29
4        97884  19.78
5        36995   7.48
9         4182   0.85
6         3949   0.80
7         2522   0.51
998       1779   0.36

--- ABO ---
      count    pct
ABO               
O    243693  49.24
A    155594  31.44
B     74036  14.96
AB    18475   3.73
A1     2398   0.48
A2      365   0.07
A1B     226   0.05
A2B      75   0.02

--- ON_DIALYSIS ---
              count    pct
ON_DIALYSIS               
Y            360617  72.87
N            134245  27.13

--- INIT_STAT ---
            count    pct
INIT_STAT               
4010.0     348947  70.51
4099.0     145670  29.44
4050.0        196   0.04
4060.0         30   0.01
4020.0         19   0.00

--- REGION ---
        count    pct
REGION              
5       81578  16.48
3       69493  14.04
1

In [ ]:
# ============================================================
# D3. FUNC_STAT_TCR — special case
# Two code scales: 2010-2100 (adult) and 4010-4100 (alternate)
# Plus 996/998 for unknowns
# ============================================================

print("=== D3. FUNC_STAT_TCR ===\n")

func = kidney_waitlist_analytics['FUNC_STAT_TCR'].dropna()

# Full value counts
print("All codes present:")
print(func.value_counts().sort_index())

# Categorize into scales
adult_scale = func[(func >= 2010) & (func <= 2100)]
alt_scale   = func[(func >= 4010) & (func <= 4100)]
unknowns    = func[func.isin([996, 998])]
other       = func[~func.isin(adult_scale.index.tolist() + alt_scale.index.tolist() + [996, 998])]

print(f"\nAdult scale (2010-2100):     {len(adult_scale):,}  ({len(adult_scale)/len(func)*100:.1f}%)")
print(f"Alternate scale (4010-4100): {len(alt_scale):,}  ({len(alt_scale)/len(func)*100:.1f}%)")
print(f"Unknown (996/998):           {len(unknowns):,}  ({len(unknowns)/len(func)*100:.1f}%)")
print(f"Unexpected codes:            {len(other):,}")

if len(other) > 0:
    print(f"  ❌ Unexpected values: {other.unique()}")
else:
    print(f"  ✅ No unexpected codes")

=== D3. FUNC_STAT_TCR ===

All codes present:
FUNC_STAT_TCR
1.0            1
996.0        121
998.0       8700
2010.0       371
2020.0      2936
2030.0      1905
2040.0      8364
2050.0     23923
2060.0     36967
2070.0    111493
2080.0    139478
2090.0    109316
2100.0     35412
4010.0        43
4020.0        23
4030.0        89
4040.0       187
4050.0       157
4060.0       505
4070.0      1029
4080.0      2812
4090.0      2413
4100.0      4642
Name: count, dtype: int64

Adult scale (2010-2100):     470,165  (95.8%)
Alternate scale (4010-4100): 11,900  (2.4%)
Unknown (996/998):           8,821  (1.8%)
Unexpected codes:            0
  ✅ No unexpected codes


In [ ]:
# ============================================================
# D4. Timing columns
# INIT_DATE / END_DATE range, days_to_event distribution
# ============================================================

print("=== D4. TIMING COLUMNS ===\n")

kidney_waitlist_analytics['INIT_DATE'] = pd.to_datetime(kidney_waitlist_analytics['INIT_DATE'])
kidney_waitlist_analytics['END_DATE']  = pd.to_datetime(kidney_waitlist_analytics['END_DATE'])

print("--- INIT_DATE ---")
print(f"Min: {kidney_waitlist_analytics['INIT_DATE'].min()}")
print(f"Max: {kidney_waitlist_analytics['INIT_DATE'].max()}")
print(f"Confirms 2015+ filter: {'✅' if kidney_waitlist_analytics['INIT_DATE'].min().year >= 2015 else '❌'}")

print("\n--- END_DATE ---")
print(f"Min: {kidney_waitlist_analytics['END_DATE'].min()}")
print(f"Max: {kidney_waitlist_analytics['END_DATE'].max()}")

print("\n--- days_to_event ---")
print(kidney_waitlist_analytics['days_to_event'].describe())
dte = kidney_waitlist_analytics['days_to_event'].dropna()
neg = (dte < 0).sum()
print(f"\nNegative values: {neg}  (expected: 0 after teammate nulled 59 rows)")
print(f"Null count:      {kidney_waitlist_analytics['days_to_event'].isnull().sum()}")


=== D4. TIMING COLUMNS ===

--- INIT_DATE ---
Min: 2015-01-01 00:00:00
Max: 2026-06-30 00:00:00
Confirms 2015+ filter: ✅

--- END_DATE ---
Min: 2002-12-15 00:00:00
Max: 2026-07-03 00:00:00

--- days_to_event ---
count    494803.000000
mean        649.541492
std         654.237636
min           0.000000
25%         144.000000
50%         433.000000
75%         965.000000
max        4200.000000
Name: days_to_event, dtype: float64

Negative values: 0  (expected: 0 after teammate nulled 59 rows)
Null count:      59


DISCOVERY PROCESS DONE!

We uncovered some insights in this task, which will be documented and further analyzed and process with step 3 and 4

# TASK 2. DATA DISTRIBUTION

TO-DO Overview:

1. Split the data on INIT_DATE at 2021-01-01, print group sizes
2. Compare outcome rates across eras: did transplant rates, death rates, removal rates change?
3. Compare feature distributions: did the patient population itself change? (age, BMI, CPRA, ethnicity, blood type, dialysis, region)
4. Compare missingness: is INIT_CPRA's 28.84% gap evenly split or concentrated in one era?
5. Interpret and recommend: should the model team encode a policy_era feature, train only on post-2021, or evaluate performance separately across eras?

In [ ]:
# ============================================================
# Step 1: Split pre/post 2021
# ============================================================

# Ensure INIT_DATE is datetime (may already be from D4)
kidney_waitlist_analytics['INIT_DATE'] = pd.to_datetime(kidney_waitlist_analytics['INIT_DATE'])

split_date = pd.Timestamp('2021-01-01')

pre_2021  = kidney_waitlist_analytics[kidney_waitlist_analytics['INIT_DATE'] < split_date]
post_2021 = kidney_waitlist_analytics[kidney_waitlist_analytics['INIT_DATE'] >= split_date]

print(f"Pre-2021:  {len(pre_2021):,} rows ({len(pre_2021)/len(kidney_waitlist_analytics)*100:.1f}%)")
print(f"Post-2021: {len(post_2021):,} rows ({len(post_2021)/len(kidney_waitlist_analytics)*100:.1f}%)")
print(f"Total:     {len(pre_2021)+len(post_2021):,}")

Pre-2021:  232,376 rows (47.0%)
Post-2021: 262,486 rows (53.0%)
Total:     494,862


In [ ]:
# ============================================================
# Step 2: Outcome rates by era
# ============================================================

pre_rates  = pre_2021['outcome'].value_counts(normalize=True).round(4) * 100
post_rates = post_2021['outcome'].value_counts(normalize=True).round(4) * 100

comparison = pd.DataFrame({
    'pre_2021_%':  pre_rates,
    'post_2021_%': post_rates,
})
comparison['shift'] = comparison['post_2021_%'] - comparison['pre_2021_%']
comparison = comparison.sort_values('pre_2021_%', ascending=False)

print("=== OUTCOME RATES BY ERA ===\n")
print(comparison.round(2))

# event_adverse comparison
pre_adverse  = pre_2021['event_adverse'].mean() * 100
post_adverse = post_2021['event_adverse'].mean() * 100
print(f"\nevent_adverse:  pre={pre_adverse:.2f}%  post={post_adverse:.2f}%  shift={post_adverse - pre_adverse:+.2f}pp")

=== OUTCOME RATES BY ERA ===

                        pre_2021_%  post_2021_%  shift
outcome                                               
transplanted                 54.30        41.24 -13.06
removed_administrative       12.29         6.39  -5.90
died                         10.35         3.99  -6.36
removed_too_sick              9.79         4.75  -5.04
transplanted_elsewhere        9.11         6.13  -2.98
still_waiting                 2.87        36.82  33.95
unknown                       1.30         0.68  -0.62

event_adverse:  pre=20.14%  post=8.74%  shift=-11.39pp


In [ ]:
# ============================================================
# Step 3a: Numerical features by era
# ============================================================

print("=== NUMERICAL FEATURES BY ERA ===\n")

for col in ['INIT_AGE', 'BMI_TCR', 'INIT_CPRA']:
    pre_desc  = pre_2021[col].describe()
    post_desc = post_2021[col].describe()
    print(f"--- {col} ---")
    print(f"  Pre-2021:  mean={pre_desc['mean']:.2f}  median={pre_desc['50%']:.2f}")
    print(f"  Post-2021: mean={post_desc['mean']:.2f}  median={post_desc['50%']:.2f}")
    print(f"  Mean shift: {post_desc['mean'] - pre_desc['mean']:+.2f}")
    print()

=== NUMERICAL FEATURES BY ERA ===

--- INIT_AGE ---
  Pre-2021:  mean=51.32  median=54.00
  Post-2021: mean=52.37  median=55.00
  Mean shift: +1.05

--- BMI_TCR ---
  Pre-2021:  mean=41.57  median=28.41
  Post-2021: mean=85.20  median=28.65
  Mean shift: +43.63

--- INIT_CPRA ---
  Pre-2021:  mean=6.36  median=0.00
  Post-2021: mean=16.44  median=0.00
  Mean shift: +10.08



In [ ]:
# ============================================================
# Step 3b: Categorical features by era
# ============================================================

print("=== CATEGORICAL FEATURES BY ERA ===\n")

for col in ['ETHCAT', 'ABO', 'ON_DIALYSIS', 'REGION']:
    pre_pct  = pre_2021[col].value_counts(normalize=True).round(4) * 100
    post_pct = post_2021[col].value_counts(normalize=True).round(4) * 100
    comp = pd.DataFrame({
        'pre_%':  pre_pct,
        'post_%': post_pct,
    })
    comp['shift'] = comp['post_%'] - comp['pre_%']
    comp = comp.sort_values('pre_%', ascending=False)
    print(f"--- {col} ---")
    print(comp.round(2))
    print()

=== CATEGORICAL FEATURES BY ERA ===

--- ETHCAT ---
        pre_%  post_%  shift
ETHCAT                      
1       43.14   39.00  -4.14
2       28.39   30.08   1.69
4       18.94   20.53   1.59
5        7.43    7.52   0.09
6        0.82    0.77  -0.05
9        0.80    0.89   0.09
7        0.48    0.53   0.05
998      0.00    0.68   0.68

--- ABO ---
     pre_%  post_%  shift
ABO                      
O    48.87   49.57   0.70
A    31.76   31.16  -0.60
B    14.87   15.04   0.17
AB    3.73    3.74   0.01
A1    0.60    0.38  -0.22
A2    0.09    0.06  -0.03
A1B   0.06    0.04  -0.02
A2B   0.02    0.01  -0.01

--- ON_DIALYSIS ---
             pre_%  post_%  shift
ON_DIALYSIS                      
Y            75.85   70.24  -5.61
N            24.15   29.76   5.61

--- REGION ---
        pre_%  post_%  shift
REGION                      
5       17.12   15.92  -1.20
3       13.91   14.16   0.25
2       12.94   11.32  -1.62
4       11.36   11.47   0.11
11      10.79   13.30   2.51
7        

In [ ]:
# ============================================================
# Task 2, Step 4: Missingness by era
# ============================================================

print("=== MISSINGNESS BY ERA ===\n")

cols_to_check = ['INIT_CPRA', 'BMI_TCR', 'FUNC_STAT_TCR', 'DIALYSIS_DATE', 'DAYSWAIT_ALLOC']

for col in cols_to_check:
    pre_miss  = pre_2021[col].isnull().mean() * 100
    post_miss = post_2021[col].isnull().mean() * 100
    print(f"{col:<20} pre={pre_miss:.2f}%  post={post_miss:.2f}%  shift={post_miss - pre_miss:+.2f}pp")

=== MISSINGNESS BY ERA ===

INIT_CPRA            pre=0.00%  post=54.38%  shift=+54.38pp
BMI_TCR              pre=0.25%  post=0.37%  shift=+0.11pp
FUNC_STAT_TCR        pre=0.07%  post=1.46%  shift=+1.39pp
DIALYSIS_DATE        pre=24.15%  post=29.76%  shift=+5.60pp
DAYSWAIT_ALLOC       pre=1.46%  post=1.28%  shift=-0.17pp
